# BeliefLens SPY/SGOV authenticated workflow

This notebook is the configurable companion to the frozen public reproduction. It shows how to create a private semantic-state benchmark from your own labelled evidence, freeze its partitions and prompt instrument, obtain an API-cost estimate, explicitly approve a model run, retrieve calibration diagnostics and apply the resulting state probabilities to a user-owned strategy.

## Credentials

1. [Request a BeliefLens API key](https://demo.belieflens.org/signup).
2. Set `BELIEFLENS_API_KEY` in your environment.
3. Set your own `OPENAI_API_KEY` only when you are ready to execute the estimated job.

Your provider key is transmitted only with the approved job request and is not written into this notebook or the reproducibility archive. Never commit credentials. This workflow creates resources in your private BeliefLens workspace and the approved execution incurs provider charges.


In [ ]:
from pathlib import Path
from io import BytesIO
import json, os, time, uuid, zipfile
import httpx
import numpy as np
import pandas as pd

pd.set_option('display.max_colwidth', 100)
BELIEFLENS_URL = os.getenv('BELIEFLENS_URL', 'https://demo.belieflens.org')
BELIEFLENS_API_KEY = os.getenv('BELIEFLENS_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
CUSTOM_MODEL = os.getenv('BELIEFLENS_MODEL', 'gpt-4.1-mini')
CUSTOM_RECORDS_CSV = os.getenv('BELIEFLENS_RECORDS_CSV')

bundle_candidates = [
    Path.cwd()/'data/offline_reproduction',
    Path.cwd()/'examples/notebooks/finance/data/offline_reproduction',
]
BUNDLE = next((path.resolve() for path in bundle_candidates if path.exists()), bundle_candidates[0])
assert BUNDLE.exists(), f'Frozen schema-example bundle not found: {BUNDLE}'
print('BeliefLens key configured:', bool(BELIEFLENS_API_KEY))
print('Provider key configured:', bool(OPENAI_API_KEY))


## 1. Prepare your benchmark

Supply a CSV using `BELIEFLENS_RECORDS_CSV`. It must contain `evidence_text`, `source_id`, and `reference_state`. Recommended provenance includes observation time, retrieval time, target, horizon, and review status. If no CSV is supplied, the frozen records are loaded only as a schema illustration.


In [ ]:
schema_records = pd.DataFrame(
    json.loads(line)
    for line in (BUNDLE/'inputs/records.jsonl').read_text().splitlines()
    if line.strip()
).rename(columns={'observation_id': 'source_id'})

custom_records = schema_records.copy() if not CUSTOM_RECORDS_CSV else pd.read_csv(CUSTOM_RECORDS_CSV)
required = {'evidence_text', 'source_id', 'reference_state'}
assert required <= set(custom_records), f'Missing columns: {sorted(required-set(custom_records))}'
custom_records[list(required) + [c for c in ['decision_date', 'partition'] if c in custom_records]].head(3)


## 2. Define and freeze the measurement experiment

The next cell defines helper functions. The final two lines remain commented so inspection is safe: uncommenting them creates a private world, dataset and frozen prompt instrument, uploads records, freezes partitions, and requests a cost estimate. It does **not** yet make a provider call.


In [ ]:
def api(method, path, *, json_body=None, extra_headers=None, timeout=180):
    if not BELIEFLENS_API_KEY: raise RuntimeError('Set BELIEFLENS_API_KEY before using the configurable workflow')
    headers = {'Authorization': f'Bearer {BELIEFLENS_API_KEY}'} | (extra_headers or {})
    response = httpx.request(method, BELIEFLENS_URL.rstrip('/') + path, headers=headers, json=json_body, timeout=timeout)
    response.raise_for_status()
    return response

def create_custom_experiment(frame):
    states = ['Risk-on', 'Mixed', 'Risk-off', 'Unsure']
    world = api('POST', '/v1/worlds', json_body={
        'name': f'Custom SPY state study {uuid.uuid4().hex[:8]}', 'domain': 'finance',
        'target': 'US broad-equity market state', 'horizon': 'declared by each evidence record', 'states': states,
        'state_definitions': {'Risk-on': 'Evidence supports broad-equity risk taking.', 'Mixed': 'Evidence is materially offsetting.',
                              'Risk-off': 'Evidence supports defensive positioning.', 'Unsure': 'Evidence is insufficient.'}
    }).json()
    dataset = api('POST', '/v1/datasets', json_body={'world_id': world['world_id'], 'name': 'Custom evidence', 'visibility': 'private_pilot', 'review_mode': 'manual'}).json()
    upload = frame.where(pd.notna(frame), None).to_dict('records')
    for start in range(0, len(upload), 250):
        api('POST', f"/v1/datasets/{dataset['id']}/observations", json_body={'records': upload[start:start+250]})
    api('POST', f"/v1/datasets/{dataset['id']}/partitions", json_body={'seed': 20260821, 'proportions': {'calibration': .4, 'prompt_validation': .2, 'conformal': .2, 'test': .2}})
    template = 'States:\n{state_definitions}\nEvidence:\n{evidence}\nAllowed responses:\n{verbalizers}'
    variants = [{'variant_id': 'evidence-first', 'template': 'Evidence:\n{evidence}\nStates:\n{state_definitions}\nAllowed responses:\n{verbalizers}'}]
    prompt = api('POST', '/v1/prompts', json_body={'world_id': world['world_id'], 'name': 'SPY semantic instrument', 'template': template,
        'verbalizers': {'Risk-on': 'Bull', 'Mixed': 'Mixed', 'Risk-off': 'Bear', 'Unsure': 'Unsure'}, 'variants': variants, 'output_contract': 'Return exactly one declared response.', 'minimum_expressed_mass': .8}).json()
    api('POST', f"/v1/datasets/{dataset['id']}/freeze"); api('POST', f"/v1/prompts/{prompt['id']}/freeze")
    estimate = api('POST', '/v1/prompt-experiments/estimates', json_body={'dataset_id': dataset['id'], 'prompt_id': prompt['id'], 'model': CUSTOM_MODEL, 'repeats': 1, 'concurrency': 4}).json()
    return world, dataset, prompt, estimate

# This mutates your private BeliefLens workspace but makes no OpenAI call. Uncomment when ready.
# world, dataset, prompt, estimate = create_custom_experiment(custom_records)
# pd.Series(estimate)

## 3. Inspect cost, then explicitly approve execution

Review `estimate` before executing. `approve_and_run` sends the provider credential only with the approved job, prints an ASCII progress indicator, downloads the completed archive, and exposes its certificate and held-out measurements. This is the only stage that incurs model-provider charges.


In [ ]:
def approve_and_run(estimate):
    if not OPENAI_API_KEY: raise RuntimeError('Set OPENAI_API_KEY; BeliefLens does not provide or store it')
    high = float(estimate['estimated_cost_usd']['high'])
    print(f"Approving a maximum estimated provider cost of ${high:.4f}")
    job = api('POST', '/v1/jobs', json_body={'estimate_id': estimate['estimate_id'], 'hard_budget_usd': max(.01, high)},
              extra_headers={'X-OpenAI-API-Key': OPENAI_API_KEY, 'Idempotency-Key': f'notebook-{uuid.uuid4()}' }).json()
    while True:
        status = api('GET', f"/v1/jobs/{job['id']}").json()
        print(f"\r[{status['percent']:6.2f}%] {status['phase']}", end='')
        if status['status'] in {'completed', 'failed', 'cancelled'}: break
        time.sleep(3)
    print()
    if status['status'] != 'completed': raise RuntimeError(status)
    return api('GET', f"/v1/jobs/{job['id']}/result").content

# This is the only step that incurs OpenAI charges. Inspect `estimate` first, then uncomment.
# custom_archive = approve_and_run(estimate)
# with zipfile.ZipFile(BytesIO(custom_archive)) as zf:
#     custom_certificate = json.loads(zf.read('certificate.json'))
#     custom_measurements = pd.read_csv(zf.open('held_out_results.csv'))
# pd.Series({key: custom_certificate[key] for key in ['certificate_status', 'accuracy', 'mean_js_error', 'empirical_test_coverage', 'mean_prompt_perturbation_js']})

## 4. Use calibrated states in a separately validated strategy

The example mapping below is intentionally downstream of semantic calibration. You own and must validate the mapping from state probabilities to positions, the point-in-time return alignment, transaction costs and investment assumptions. Semantic calibration does not certify trading profitability.


In [ ]:
def apply_strategy(calibrated, market_returns, *, transaction_cost_bps=5):
    """Example only: users own and validate the mapping from state probabilities to positions."""
    probabilities = np.vstack(calibrated['calibrated_state_probabilities'].map(lambda value: json.loads(value) if isinstance(value, str) else value))
    spy_weight = probabilities[:, 0] + .5 * probabilities[:, 1]
    turnover = np.abs(np.diff(spy_weight, prepend=spy_weight[0]))
    returns = spy_weight * market_returns['SPY_return'].to_numpy() + (1-spy_weight) * market_returns['safe_return'].to_numpy() - turnover * transaction_cost_bps/10_000
    return pd.DataFrame({'SPY_weight': spy_weight, 'strategy_return': returns}, index=market_returns.index)

# Supplied Massive-derived history used by the demonstration; replace it with your own point-in-time data if desired.
historical_prices = pd.read_csv(BUNDLE/'inputs/prices_total_return.csv', index_col=0, parse_dates=True)
historical_prices[['SPY', 'SGOV']].tail()
# Your merge must align each untouched-test measurement with returns occurring strictly after its observation time.
# strategy_results = apply_strategy(custom_measurements, your_aligned_future_returns)

## Audit boundary

BeliefLens qualifies a declared measurement channel: it records the benchmark specification, frozen partitions, prompt version, provider observations, calibration diagnostics, uncertainty coverage and execution provenance. The portfolio rule remains a separate decision layer. Preserve the returned certificate and archive with the code and market-data vintage used in your evaluation.
